# Iran Shock & GB Power — Dashboard

Four-panel dashboard mirroring the Tableau Public layout, built in plotly so it can also be served as a static HTML page from GitHub Pages.

**Mental model.** Crude is the signal, gas is the mechanism, cash-out is the amplifier (see `docs/00_findings.md` for the transmission chain). The hero chart shows Brent (oil signal), TTF (European gas — the mechanism), and GB MID (the downstream effect) on one normalised axis.

**Data flow.** This notebook is pure analysis — it reads from DuckDB views built by `notebooks/05_build_views.ipynb`. To refresh the underlying data, run `uv run iran-shock-ingest-prices` and `uv run iran-shock-ingest-gdelt`, then re-execute notebook 05.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from iran_shock.config import DUCKDB_PATH

con = duckdb.connect(DUCKDB_PATH, read_only=True)

combined = con.execute("SELECT * FROM v_combined_daily ORDER BY d").df()
events_daily = con.execute("SELECT * FROM v_middle_east_events ORDER BY event_date").df()
impact = con.execute("SELECT * FROM v_event_impact ORDER BY event_date").df()
geo = con.execute("SELECT * FROM v_event_geo").df()
timeline = con.execute("SELECT * FROM timeline ORDER BY event_date").df()

print("v_combined_daily:    ", combined.shape)
print("v_middle_east_events:", events_daily.shape)
print("v_event_impact:      ", impact.shape)
print("v_event_geo:         ", geo.shape)
print("timeline:            ", timeline.shape)

v_combined_daily:     (102, 6)
v_middle_east_events: (150, 5)
v_event_impact:       (15, 11)
v_event_geo:          (75113, 7)
timeline:             (15, 5)


## Panel 1 — Hero: Brent, TTF gas, GB power (% from 1 Feb)

Three series on a single axis, normalised to % change from the first available date. Vertical reference lines mark the 15 curated timeline events. Reading this chart: where the three series move together, the transmission chain is working (Iran shock → crude → gas → power). Where GB power decouples from Brent but tracks TTF, gas-as-marginal-fuel is doing the work.

In [3]:
df = combined.copy()
for col in ("brent_close", "ttf_close", "gb_power_avg"):
    first = df[col].dropna().iloc[0]
    df[f"{col}_pct"] = (df[col] / first - 1) * 100

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=df["d"], y=df["brent_close_pct"], name="Brent (oil signal)", line=dict(width=2))
)
fig.add_trace(
    go.Scatter(x=df["d"], y=df["ttf_close_pct"], name="TTF gas (mechanism)", line=dict(width=2))
)
fig.add_trace(
    go.Scatter(
        x=df["d"], y=df["gb_power_avg_pct"], name="GB MID power (downstream)", line=dict(width=2)
    )
)

for _, e in timeline.iterrows():
    fig.add_vline(x=e["event_date"], line_dash="dot", line_color="grey", opacity=0.4)

fig.update_layout(
    title="Brent · TTF · GB power — % change from 1 Feb 2026",
    yaxis_title="% change from 1 Feb",
    xaxis_title="",
    template="plotly_white",
    hovermode="x unified",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

## Panel 2 — News signal: GDELT daily volume and tone

Daily count of Middle East conflict events (bars) and the mean tone score (line). The pre-war baseline is already 2,000–4,000 events/day across the 12 FIPS countries; the war shock is the *jump above* that baseline. Tone runs persistently negative — read the sign of changes, not the absolute level.

In [4]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(
        x=events_daily["event_date"],
        y=events_daily["event_count"],
        name="Event count",
        marker_color="lightsteelblue",
        opacity=0.75,
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=events_daily["event_date"],
        y=events_daily["avg_tone"],
        name="Avg tone",
        line=dict(color="firebrick", width=2),
    ),
    secondary_y=True,
)

for _, e in timeline.iterrows():
    fig.add_vline(x=e["event_date"], line_dash="dot", line_color="grey", opacity=0.4)

fig.update_yaxes(title_text="Conflict event count", secondary_y=False)
fig.update_yaxes(title_text="Avg tone (more negative = lower)", secondary_y=True)
fig.update_layout(
    title="Middle East conflict news — daily volume and tone",
    template="plotly_white",
    hovermode="x unified",
    height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

## Panel 3 — Per-event impact at +3 trading days

For each curated timeline event: Brent, TTF, and GB MID prices on the event day (`t`) and three trading days later (`t+3`), plus the % move. Uses nearest-trading-day lookup so weekend events still produce clean numbers. The three % columns side-by-side are the transmission test — where Brent and TTF moved together with GB power, the chain is visible; where TTF moved but Brent didn't (e.g. Qatar LNG strikes), gas-as-mechanism stands out.

In [5]:
df = impact.copy()


def _fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{x:+.1f}%"


def _fmt(x, d=2):
    if pd.isna(x):
        return ""
    return f"{x:.{d}f}"


def _pct_colour(x):
    if pd.isna(x):
        return "#f5f5f5"
    return "#ffe5e5" if x < 0 else "#e5f5e5"


brent_colours = [_pct_colour(v) for v in df["brent_pct_3d"]]
ttf_colours = [_pct_colour(v) for v in df["ttf_pct_3d"]]
power_colours = [_pct_colour(v) for v in df["power_pct_3d"]]
neutral = ["white"] * len(df)

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[90, 220, 60, 60, 70, 60, 60, 70, 60, 60, 70],
            header=dict(
                values=[
                    "Date",
                    "Event",
                    "Brent t",
                    "Brent t+3",
                    "Brent %",
                    "TTF t",
                    "TTF t+3",
                    "TTF %",
                    "Power t",
                    "Power t+3",
                    "Power %",
                ],
                fill_color="#333",
                font=dict(color="white", size=12),
                align="left",
            ),
            cells=dict(
                values=[
                    pd.to_datetime(df["event_date"]).dt.strftime("%Y-%m-%d"),
                    df["label"],
                    df["brent_t"].map(_fmt),
                    df["brent_t3"].map(_fmt),
                    df["brent_pct_3d"].map(_fmt_pct),
                    df["ttf_t"].map(_fmt),
                    df["ttf_t3"].map(_fmt),
                    df["ttf_pct_3d"].map(_fmt_pct),
                    df["power_t"].map(_fmt),
                    df["power_t3"].map(_fmt),
                    df["power_pct_3d"].map(_fmt_pct),
                ],
                fill_color=[
                    neutral,
                    neutral,
                    neutral,
                    neutral,
                    brent_colours,
                    neutral,
                    neutral,
                    ttf_colours,
                    neutral,
                    neutral,
                    power_colours,
                ],
                align="left",
                font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title="Per-event impact at +3 trading days", height=560, margin=dict(t=50, l=10, r=10, b=10)
)
fig.show()

## Panel 4 — Conflict-event map (Middle East)

Point map from `v_event_geo`. Filtered to city/landmark resolution (`ActionGeo_Type` 3/4) so points sit on real places instead of country centroids. Size by event count, colour by average tone (more red = more negative). Framed on the Middle East.

In [6]:
df = geo.copy()
# Drop one-off / noisy single-event places to make the chart legible.
df = df[df["event_count"] >= 5]

fig = px.scatter_geo(
    df,
    lat="lat",
    lon="lon",
    size="event_count",
    size_max=30,
    color="avg_tone",
    color_continuous_scale="RdBu",
    range_color=[-8, -1],
    hover_name="place",
    hover_data={
        "event_count": True,
        "avg_tone": ":.2f",
        "total_mentions": True,
        "lat": False,
        "lon": False,
    },
    projection="natural earth",
    title="Conflict event locations — Feb to May 2026 (size = event count, colour = avg tone)",
)
fig.update_geos(
    lonaxis_range=[20, 70],
    lataxis_range=[10, 45],
    showcountries=True,
    countrycolor="#bbbbbb",
    showcoastlines=True,
    coastlinecolor="#888888",
    showland=True,
    landcolor="#f5f5f5",
    showocean=True,
    oceancolor="#e8eef5",
)
fig.update_layout(height=550, margin=dict(t=50, l=10, r=10, b=10))
fig.show()

## Panel 5 — Robustness statistics

With n=15 (14 usable; the 28 May row has NaN for t+3 because today is 30 May), inferential statistics are suggestive at best. The point of this section is to be honest about which correlations survive a rank-based test, drop a single influential point, or hold up under resampling — and which don't. The headline survivor is the cross-commodity Brent↔TTF result; everything else needs context.

In [7]:
import numpy as np

# Join event-impact with the news metrics on event_date
impact = con.execute("SELECT * FROM v_event_impact ORDER BY event_date").df()
news = con.execute("SELECT * FROM v_middle_east_events").df()

stats_df = impact.merge(news, on="event_date", how="left")
stats_df = stats_df.dropna(subset=["brent_pct_3d", "ttf_pct_3d", "power_pct_3d"])
print(f"n usable events: {len(stats_df)}")
stats_df[
    [
        "event_date",
        "label",
        "brent_pct_3d",
        "ttf_pct_3d",
        "power_pct_3d",
        "event_count",
        "avg_tone",
        "avg_goldstein",
    ]
]

n usable events: 14


,event_date,label,brent_pct_3d,ttf_pct_3d,power_pct_3d,event_count,avg_tone,avg_goldstein
0,2026-02-28,Operation Epic Fury begins,4.7,22.0,61.4,22754,-4.557167,-6.382236
1,2026-03-01,Iran retaliates; first US dead,4.7,9.6,67.2,24257,-4.828647,-6.701608
2,2026-03-05,Hormuz paralysis starts,15.9,11.3,26.1,26764,-4.618060,-6.155463
3,2026-03-07,Israeli strikes hit Tehran/Alborz fuel depots,-11.3,-16.0,-32.5,16414,-4.793029,-6.478799
4,2026-03-12,KC-135 loss and tanker attacks,-0.2,0.0,16.6,21902,-4.742384,-6.216889
5,2026-03-18,Iran strikes Qatar LNG infrastructure,-6.9,3.7,11.1,21840,-4.668429,-6.228530
6,2026-03-24,US 15-point settlement proposal,7.7,0.3,70.6,18138,-4.473392,-6.080637
7,2026-03-31,Iran strikes loaded tanker off Dubai,-7.2,-1.4,-15.4,18581,-4.864865,-6.060847
8,2026-04-02,Trump vows harder phase of attacks,0.7,0.0,-4.0,16723,-4.595288,-5.843365
9,2026-04-07,Pakistan-brokered ceasefire announced,-12.9,-18.0,3.7,18130,-4.975196,-5.983486


### Pearson vs Spearman correlations

Pearson assumes linear relationships and is sensitive to outliers; Spearman is rank-based and robust. If a result holds in both, it's real signal. If only Pearson is significant, treat it as outlier-driven.

In [8]:
cols = ["brent_pct_3d", "ttf_pct_3d", "power_pct_3d", "event_count", "avg_tone", "avg_goldstein"]
print("Pearson r:")
print(stats_df[cols].corr(method="pearson").round(2))
print()
print("Spearman ρ (rank-based, robust to outliers):")
print(stats_df[cols].corr(method="spearman").round(2))

Pearson r:
               brent_pct_3d  ttf_pct_3d  power_pct_3d  event_count  avg_tone  \
brent_pct_3d           1.00        0.75          0.59         0.34      0.53   
ttf_pct_3d             0.75        1.00          0.67         0.49      0.50   
power_pct_3d           0.59        0.67          1.00         0.50      0.37   
event_count            0.34        0.49          0.50         1.00     -0.16   
avg_tone               0.53        0.50          0.37        -0.16      1.00   
avg_goldstein         -0.06       -0.28         -0.36        -0.66      0.30   

               avg_goldstein  
brent_pct_3d           -0.06  
ttf_pct_3d             -0.28  
power_pct_3d           -0.36  
event_count            -0.66  
avg_tone                0.30  
avg_goldstein           1.00  

Spearman ρ (rank-based, robust to outliers):
               brent_pct_3d  ttf_pct_3d  power_pct_3d  event_count  avg_tone  \
brent_pct_3d           1.00        0.83          0.62         0.23      0.48   
ttf_p

### Bootstrap CI on the headline result (Brent↔TTF)

Brent↔TTF was the only correlation with real statistical power in the previous analysis (r ≈ +0.75, p ≈ 0.002). Bootstrap resampling tells us how stable it is at this sample size — narrow CI means the result is robust to sampling variation; wide CI means it could just as easily be much weaker.

In [9]:
rng = np.random.default_rng(42)
n_boot = 5000
n = len(stats_df)
brent = stats_df["brent_pct_3d"].values
ttf = stats_df["ttf_pct_3d"].values

r_obs = float(np.corrcoef(brent, ttf)[0, 1])
boot_rs = []
for _ in range(n_boot):
    idx = rng.choice(n, n, replace=True)
    if np.std(brent[idx]) > 0 and np.std(ttf[idx]) > 0:
        boot_rs.append(np.corrcoef(brent[idx], ttf[idx])[0, 1])
boot_rs = np.array(boot_rs)

print(f"Brent↔TTF Pearson r (observed): {r_obs:+.3f}")
print(
    f"Bootstrap 95% CI:                [{np.percentile(boot_rs, 2.5):+.3f}, {np.percentile(boot_rs, 97.5):+.3f}]"
)
print(f"P(r > 0):                        {(boot_rs > 0).mean() * 100:.1f}%")
print(f"P(r > 0.5):                      {(boot_rs > 0.5).mean() * 100:.1f}%")

Brent↔TTF Pearson r (observed): +0.747


Bootstrap 95% CI:                [+0.483, +0.914]
P(r > 0):                        100.0%
P(r > 0.5):                      97.0%


### Leave-one-out

Drop each row in turn, recompute the correlation, and report the full-sample value plus the min/max across the 14 leave-one-out samples. If the min and max bracket zero, a single point is driving the result and the correlation isn't real. A `⚠` flag in the last column means the sign of the correlation flipped under at least one drop.

In [10]:
pairs = [
    ("brent_pct_3d", "ttf_pct_3d"),
    ("ttf_pct_3d", "power_pct_3d"),
    ("brent_pct_3d", "power_pct_3d"),
    ("avg_tone", "brent_pct_3d"),
    ("avg_tone", "ttf_pct_3d"),
    ("avg_tone", "power_pct_3d"),
    ("avg_goldstein", "power_pct_3d"),
    ("event_count", "ttf_pct_3d"),
]

print(f"{'pair':<35} {'full':>7} {'min LOO':>9} {'max LOO':>9} {'flag':>6}")
print("-" * 70)
for a, b in pairs:
    full = float(stats_df[a].corr(stats_df[b]))
    loo = []
    for i in stats_df.index:
        sub = stats_df.drop(i)
        loo.append(float(sub[a].corr(sub[b])))
    sign_flip = (np.sign(min(loo)) != np.sign(full)) or (np.sign(max(loo)) != np.sign(full))
    flag = "⚠" if sign_flip else ""
    print(f"{a + ' ↔ ' + b:<35} {full:>+7.2f} {min(loo):>+9.2f} {max(loo):>+9.2f} {flag:>6}")

pair                                   full   min LOO   max LOO   flag
----------------------------------------------------------------------
brent_pct_3d ↔ ttf_pct_3d             +0.75     +0.68     +0.80       
ttf_pct_3d ↔ power_pct_3d             +0.67     +0.59     +0.78       
brent_pct_3d ↔ power_pct_3d           +0.59     +0.51     +0.65       
avg_tone ↔ brent_pct_3d               +0.53     +0.38     +0.67       
avg_tone ↔ ttf_pct_3d                 +0.50     +0.30     +0.61       
avg_tone ↔ power_pct_3d               +0.37     +0.23     +0.59       
avg_goldstein ↔ power_pct_3d          -0.36     -0.63     -0.13       
event_count ↔ ttf_pct_3d              +0.49     +0.42     +0.65       


### Multi-horizon persistence (t+1, t+3, t+5, t+10)

Reads from `v_event_impact_multi`. Mean across all events at each horizon, per series. The interpretation:
- **Brent rising through to t+10** → real supply-destruction repricing (PDF phase 2), not just sentiment.
- **TTF rising through to t+10** → gas channel sustained; transmission story intact at horizon.
- **Power dispersion across horizons** → coverage / weekend-anchoring noise that the v1 design choices can't fully suppress.

In [11]:
multi = con.execute("SELECT * FROM v_event_impact_multi ORDER BY event_date").df()
# drop the last event where t+10 isn't yet realised
multi = multi.dropna(subset=["brent_pct_10d"])
print(f"n events with full t+10 horizon: {len(multi)}")
print()
horizons = ["1d", "3d", "5d", "10d"]
out = pd.DataFrame(
    {
        "horizon": horizons,
        "brent_mean": [multi[f"brent_pct_{h}"].mean() for h in horizons],
        "brent_median": [multi[f"brent_pct_{h}"].median() for h in horizons],
        "ttf_mean": [multi[f"ttf_pct_{h}"].mean() for h in horizons],
        "ttf_median": [multi[f"ttf_pct_{h}"].median() for h in horizons],
        "power_mean": [multi[f"power_pct_{h}"].mean() for h in horizons],
        "power_median": [multi[f"power_pct_{h}"].median() for h in horizons],
    }
).round(1)
print(out.to_string(index=False))

n events with full t+10 horizon: 14

horizon  brent_mean  brent_median  ttf_mean  ttf_median  power_mean  power_median
     1d        -2.2           0.0      -1.2        -0.2         2.3           0.0
     3d        -0.2           0.2       0.9         0.2        16.3          10.7
     5d         1.6           0.8       1.0         1.4         6.5          -1.7
    10d         2.3           4.8      -1.2         0.3         1.3          -3.6


## Panel 6 — Conditional-transmission test (gas-as-marginal-fuel)

The gas-as-marginal-fuel hypothesis (per `00_findings.md`): Brent / TTF → GB power transmission is **conditional** on gas being the marginal fuel that day. When wind is high and gas is crowded down the merit order, GB power decouples from European gas. When demand is high and gas is at the margin, GB power tracks TTF tightly.

Test design: bucket the 14 usable timeline events by GB gas-share on event day (`high_gas` ≥ 30% vs `low_gas` < 30%, splitting near the 150-day median of 34.7%). Compare within-regime correlations between TTF and GB power.

Source: Elexon BMRS `/datasets/FUELHH` aggregated to daily gas share (`v_gb_gen_mix_daily`); the regime-tagged event impacts in `v_event_impact_regime`.

In [12]:
# Gas-share distribution across the full 150-day window — sanity check the threshold
mix = con.execute("SELECT * FROM v_gb_gen_mix_daily ORDER BY price_date").df()
print("Daily gas share of GB transmission-connected generation (n=150 days):")
print(mix[["gas_share_pct", "renewables_share_pct", "nuclear_share_pct"]].describe().round(1))


Daily gas share of GB transmission-connected generation (n=150 days):
       gas_share_pct  renewables_share_pct  nuclear_share_pct
count          151.0                 151.0              151.0
mean            34.2                  39.3               17.2
std             13.6                  15.2                5.7
min              8.6                  10.6               10.0
25%             22.2                  27.2               12.8
50%             34.8                  39.0               15.7
75%             45.2                  51.8               20.2
max             62.4                  68.9               35.2


In [13]:
# Regime view: each event tagged with gas share on event-day and bucketed
regime_df = con.execute("SELECT * FROM v_event_impact_regime ORDER BY event_date").df()
regime_clean = regime_df.dropna(subset=["power_pct_3d", "brent_pct_3d", "ttf_pct_3d"]).copy()
print(f"n usable events: {len(regime_clean)}")
regime_clean[
    ["event_date", "label", "gas_share_pct", "regime", "brent_pct_3d", "ttf_pct_3d", "power_pct_3d"]
]

n usable events: 14


,event_date,label,gas_share_pct,regime,brent_pct_3d,ttf_pct_3d,power_pct_3d
0,2026-02-28,Operation Epic Fury begins,23.3,low_gas,4.7,22.0,61.4
1,2026-03-01,Iran retaliates; first US dead,23.3,low_gas,4.7,9.6,67.2
2,2026-03-05,Hormuz paralysis starts,35.8,high_gas,15.9,11.3,26.1
3,2026-03-07,Israeli strikes hit Tehran/Alborz fuel depots,58.7,high_gas,-11.3,-16.0,-32.5
4,2026-03-12,KC-135 loss and tanker attacks,16.7,low_gas,-0.2,0.0,16.6
5,2026-03-18,Iran strikes Qatar LNG infrastructure,38.5,high_gas,-6.9,3.7,11.1
6,2026-03-24,US 15-point settlement proposal,12.7,low_gas,7.7,0.3,70.6
7,2026-03-31,Iran strikes loaded tanker off Dubai,40.9,high_gas,-7.2,-1.4,-15.4
8,2026-04-02,Trump vows harder phase of attacks,35.8,high_gas,0.7,0.0,-4.0
9,2026-04-07,Pakistan-brokered ceasefire announced,13.5,low_gas,-12.9,-18.0,3.7


### Mean impact by regime

Power moves are large on **low-gas** events (mean +42.6%) but coupled to gas/oil on **high-gas** events (small means, in line with TTF/Brent). Two effects compose: on low-gas days the gas channel is muted, so power moves on independent drivers (base-rate, wind, weekend coverage); on high-gas days power moves in sync with gas/oil but the gas/oil moves themselves were smaller in this window.

In [14]:
agg = (
    regime_clean.groupby("regime")
    .agg(
        n=("label", "count"),
        gas_share_mean=("gas_share_pct", "mean"),
        brent_mean=("brent_pct_3d", "mean"),
        brent_median=("brent_pct_3d", "median"),
        ttf_mean=("ttf_pct_3d", "mean"),
        ttf_median=("ttf_pct_3d", "median"),
        power_mean=("power_pct_3d", "mean"),
        power_median=("power_pct_3d", "median"),
    )
    .round(1)
)
print(agg.to_string())

          n  gas_share_mean  brent_mean  brent_median  ttf_mean  ttf_median  power_mean  power_median
regime                                                                                               
high_gas  8            42.9        -0.8          -3.4      -0.8        -0.7        -3.3          -3.4
low_gas   6            16.5         0.7           2.6       3.1         2.5        42.6          48.8


### Within-regime correlations — the headline mechanistic result

On **high-gas days**, TTF↔power tightens to ρ = +0.90 (Pearson +0.92) — near-perfect rank correlation. On **low-gas days** the same pair drops to ρ = +0.60. That's the gas-as-marginal-fuel mechanism visible in the data: when gas sets the GB merit order, European gas prices flow directly into GB power; when wind/nuclear push gas out of the stack, the transmission loosens.

Brent↔TTF is stable across regimes (cross-commodity coupling doesn't depend on the GB stack — it's set in global LNG/oil markets). The high low-gas Brent↔power (+0.99) sits on n=6 with one outlier and shouldn't be taken at face value.

In [15]:
for regime in ("high_gas", "low_gas"):
    sub = regime_clean[regime_clean["regime"] == regime]
    n = len(sub)
    print(f"\n{regime}  (n={n}, mean gas share {sub['gas_share_pct'].mean():.1f}%):")
    pairs = [
        ("brent_pct_3d", "ttf_pct_3d"),
        ("ttf_pct_3d", "power_pct_3d"),
        ("brent_pct_3d", "power_pct_3d"),
    ]
    for a, b in pairs:
        r_p = sub[a].corr(sub[b], method="pearson")
        r_s = sub[a].corr(sub[b], method="spearman")
        print(f"  {a:<14} ↔ {b:<14}  Pearson r={r_p:+.2f}  Spearman ρ={r_s:+.2f}")


high_gas  (n=8, mean gas share 42.9%):


  brent_pct_3d   ↔ ttf_pct_3d      Pearson r=+0.79  Spearman ρ=+0.81
  ttf_pct_3d     ↔ power_pct_3d    Pearson r=+0.92  Spearman ρ=+0.90
  brent_pct_3d   ↔ power_pct_3d    Pearson r=+0.72  Spearman ρ=+0.62

low_gas  (n=6, mean gas share 16.5%):
  brent_pct_3d   ↔ ttf_pct_3d      Pearson r=+0.77  Spearman ρ=+0.64
  ttf_pct_3d     ↔ power_pct_3d    Pearson r=+0.72  Spearman ρ=+0.60
  brent_pct_3d   ↔ power_pct_3d    Pearson r=+0.90  Spearman ρ=+0.99


### gas_share_pct as a continuous regressor

Same idea but without bucketing: how does each side of the transmission chain correlate with gas share on event day?

In [16]:
print(f"Correlation between event-day gas share and +3d moves (n={len(regime_clean)}):")
for col in ("brent_pct_3d", "ttf_pct_3d", "power_pct_3d"):
    r_p = regime_clean["gas_share_pct"].corr(regime_clean[col], method="pearson")
    r_s = regime_clean["gas_share_pct"].corr(regime_clean[col], method="spearman")
    print(f"  gas_share ↔ {col:<14}  Pearson r={r_p:+.2f}  Spearman ρ={r_s:+.2f}")

Correlation between event-day gas share and +3d moves (n=14):
  gas_share ↔ brent_pct_3d    Pearson r=-0.25  Spearman ρ=-0.33
  gas_share ↔ ttf_pct_3d      Pearson r=-0.28  Spearman ρ=-0.36
  gas_share ↔ power_pct_3d    Pearson r=-0.73  Spearman ρ=-0.78


### Visualising the regime split

Scatter of TTF +3d vs power +3d, coloured by regime. A clean separation by colour validates the mechanistic story visually.

In [17]:
fig = px.scatter(
    regime_clean,
    x="ttf_pct_3d",
    y="power_pct_3d",
    color="regime",
    size="gas_share_pct",
    hover_name="label",
    hover_data={
        "event_date": True,
        "gas_share_pct": ":.1f",
        "ttf_pct_3d": False,
        "power_pct_3d": False,
    },
    color_discrete_map={"high_gas": "#c0392b", "low_gas": "#2c7fb8"},
    labels={"ttf_pct_3d": "TTF % move at +3d", "power_pct_3d": "GB power % move at +3d"},
    title="Conditional transmission: TTF → GB power, split by gas regime",
)
fig.update_layout(template="plotly_white", height=500)
# zero lines
fig.add_hline(y=0, line_dash="dot", line_color="grey", opacity=0.4)
fig.add_vline(x=0, line_dash="dot", line_color="grey", opacity=0.4)
fig.show()

---

## Methodology hardening (Tier A enrichments + placebo + phase split)

Five additions on top of the headline conditional-transmission finding to harden the result and operationalise the merit-order mechanism narrative. See `docs/00_findings.md` for the consolidated numbers.

| # | Move | What it tests |
|---|---|---|
| 1.1 | News burst score (Hawkes-inspired) | Coverage spike on event days vs 30-day rolling baseline |
| 1.2 | Asymmetric tone split (TGARCH-inspired) | Does bad news drive transmission more than good news? |
| 1.3 | Rolling volatility regime | Pre-war / war / post-war regime shift across all three series |
| 1.4 | Placebo / permutation test | Real-event means vs 1,000 random non-event-day samples |
| 1.5 | Phase-aware multi-horizon split | The PDF's three-phase prediction tested at t+1, t+3, t+5, t+10 |

### News burst score (Hawkes-inspired)

`burst_score = event_count / rolling_30d_mean(event_count)`. Hawkes proper models self-exciting arrival kernels; with n=14 events that would overfit, so this is a deliberate proxy. Spike above ~1.5–2× = unusual coverage volume vs the prior 30 days.

View: `v_news_burst_daily`.

In [18]:
burst = con.execute("SELECT * FROM v_news_burst_daily ORDER BY event_date").df()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(
        x=burst["event_date"],
        y=burst["event_count"],
        name="event_count",
        opacity=0.35,
        marker=dict(color="#888"),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=burst["event_date"],
        y=burst["burst_score"],
        name="burst_score (event_count / rolling 30d mean)",
        line=dict(color="#c0392b", width=2),
    ),
    secondary_y=True,
)
fig.add_hline(y=1.0, line_dash="dot", line_color="grey", opacity=0.5, secondary_y=True)
fig.add_hline(y=2.0, line_dash="dot", line_color="red", opacity=0.4, secondary_y=True)

for _, e in timeline.iterrows():
    fig.add_vline(x=e["event_date"], line_dash="dot", line_color="grey", opacity=0.3)

fig.update_layout(
    title="News burst score — Middle-East conflict events (Hawkes-inspired)",
    xaxis_title="Date",
    template="plotly_white",
    height=380,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(title_text="event_count (bars)", secondary_y=False)
fig.update_yaxes(title_text="burst_score (line)", secondary_y=True)
fig.show()

peaks = burst.nlargest(5, "burst_score")[
    ["event_date", "event_count", "rolling_30d_mean", "burst_score"]
].round(2)
print("Top-5 burst-score dates:")
print(peaks.to_string(index=False))

Top-5 burst-score dates:
event_date  event_count  rolling_30d_mean  burst_score
2026-03-03        29364          13591.43         2.16
2026-03-02        28099          13027.90         2.16
2026-03-01        24257          12780.63         1.90
2026-03-04        26893          14305.40         1.88
2026-02-28        22754          12582.40         1.81


/var/folders/02/b59r49t147vcx84q80s5dl840000gn/T/ipykernel_49522/725869992.py:42: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


### Asymmetric tone split (TGARCH-inspired)

Splits daily Middle-East news tone into negative and positive components:

```text
negative_tone = AVG(LEAST(avgtone, 0))     -- bad-news magnitude per article
positive_tone = AVG(GREATEST(avgtone, 0))  -- good-news magnitude per article
```

Then correlates each component against the t+3 transmission. The TGARCH intuition would predict the bad-news component dominates.

In [19]:
tone_asym = con.execute("""
    SELECT a.event_date, a.negative_tone, a.positive_tone, a.overall_tone,
           b.brent_close, t.ttf_close, p.gb_power_avg
    FROM v_gdelt_tone_asymmetry_daily a
    LEFT JOIN v_brent_daily      b ON b.price_date = a.event_date
    LEFT JOIN v_ttf_daily        t ON t.price_date = a.event_date
    LEFT JOIN v_gb_power_daily   p ON p.price_date = a.event_date
    ORDER BY a.event_date
""").df()
tone_asym["brent_pct_3d"] = tone_asym["brent_close"].pct_change(3) * 100
tone_asym["ttf_pct_3d"] = tone_asym["ttf_close"].pct_change(3) * 100
tone_asym["power_pct_3d"] = tone_asym["gb_power_avg"].pct_change(3) * 100

clean = tone_asym.dropna(subset=["negative_tone", "brent_pct_3d", "ttf_pct_3d", "power_pct_3d"])
print(f"n trading-day pairs: {len(clean)}")
print()

corr_rows = []
for tone_col in ["negative_tone", "positive_tone", "overall_tone"]:
    corr_rows.append(
        {
            "tone": tone_col,
            "↔ brent t+3": round(clean[tone_col].corr(clean["brent_pct_3d"]), 3),
            "↔ ttf t+3": round(clean[tone_col].corr(clean["ttf_pct_3d"]), 3),
            "↔ power t+3": round(clean[tone_col].corr(clean["power_pct_3d"]), 3),
        }
    )
print(pd.DataFrame(corr_rows).to_string(index=False))
print()
print("Honest read: |r| < 0.30 across all three components — the asymmetric split is")
print("inconclusive at this signal granularity. AvgTone is genuinely too coarse for the")
print("test to discriminate; validates the the sentiment-depth section move to GKG + FinBERT.")

n trading-day pairs: 55

         tone  ↔ brent t+3  ↔ ttf t+3  ↔ power t+3
negative_tone       -0.064     -0.130       -0.213
positive_tone       -0.301     -0.192       -0.215
 overall_tone       -0.102     -0.147       -0.226

Honest read: |r| < 0.30 across all three components — the asymmetric split is
inconclusive at this signal granularity. AvgTone is genuinely too coarse for the
test to discriminate; validates the Phase 3 move to GKG + FinBERT.


### Rolling volatility regime

7-day rolling std of daily % moves on Brent, TTF, and power. Three regimes:

- **pre-war** (1 Jan – 27 Feb) — baseline
- **war** (28 Feb – 5 May, Epic Fury declared over)
- **post** (6 – 31 May)

In [20]:
combined_with_vol = combined.copy()
combined_with_vol["d_dt"] = pd.to_datetime(combined_with_vol["d"])
for col in ["brent_close", "ttf_close", "gb_power_avg"]:
    combined_with_vol[f"{col}_ret"] = combined_with_vol[col].pct_change() * 100
    combined_with_vol[f"{col}_vol7"] = combined_with_vol[f"{col}_ret"].rolling(7).std()

fig = go.Figure()
for col, label, colour in [
    ("brent_close", "Brent", "#e67e22"),
    ("ttf_close", "TTF", "#27ae60"),
    ("gb_power_avg", "Power", "#2980b9"),
]:
    fig.add_trace(
        go.Scatter(
            x=combined_with_vol["d_dt"],
            y=combined_with_vol[f"{col}_vol7"],
            name=f"{label} 7d vol",
            line=dict(color=colour, width=2),
        )
    )

fig.add_vrect(
    x0="2026-02-28",
    x1="2026-05-05",
    fillcolor="rgba(231,76,60,0.08)",
    line_width=0,
    annotation_text="war period",
    annotation_position="top left",
)

for _, e in timeline.iterrows():
    fig.add_vline(x=e["event_date"], line_dash="dot", line_color="grey", opacity=0.3)

fig.update_layout(
    title="Rolling 7-day std of daily % moves — regime shift across the window",
    xaxis_title="Date",
    yaxis_title="7-day std of daily % move",
    template="plotly_white",
    height=380,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

# Per-regime means
pre_war = combined_with_vol[combined_with_vol["d_dt"] < "2026-02-28"]
war = combined_with_vol[
    (combined_with_vol["d_dt"] >= "2026-02-28") & (combined_with_vol["d_dt"] <= "2026-05-05")
]
post = combined_with_vol[combined_with_vol["d_dt"] > "2026-05-05"]
print("Mean 7-day rolling std per regime:")
print(f"  {'Series':<10s}{'pre-war':>10s}{'war':>10s}{'post':>10s}")
for col, label in [("brent_close", "Brent"), ("ttf_close", "TTF"), ("gb_power_avg", "Power")]:
    print(
        f"  {label:<10s}{pre_war[col + '_vol7'].mean():>9.2f}%{war[col + '_vol7'].mean():>9.2f}%{post[col + '_vol7'].mean():>9.2f}%"
    )

Mean 7-day rolling std per regime:
  Series       pre-war       war      post
  Brent          2.29%     5.39%     3.53%
  TTF            4.95%     7.59%     3.18%
  Power         16.09%    30.09%    13.40%


### Placebo / permutation test

1,000-iteration bootstrap. For each iteration: sample 14 random non-event trading days from the eligible pool (any Brent trading day that isn't a curated event-day and has a valid t+3 lookup), compute the mean t+3 % impact for Brent / TTF / power, store. Compare the resulting placebo distribution against the real-event mean.

Code in `src/iran_shock/placebo.py`. Seeded at 42 for reproducibility.

In [21]:
from iran_shock.placebo import empirical_rank, placebo_distribution, real_event_impacts

real = real_event_impacts(con)
placebo = placebo_distribution(con, n_bootstrap=1000, seed=42)

results = []
for col, suffix in [("brent_pct_3d", "brent"), ("ttf_pct_3d", "ttf"), ("power_pct_3d", "power")]:
    real_mean = real[col].mean()
    placebo_col = placebo[f"{suffix}_pct_3d_mean"]
    results.append(
        {
            "metric (signed)": col,
            "real mean": f"{real_mean:+.2f}%",
            "placebo median": f"{placebo_col.median():+.2f}%",
            "rank": round(empirical_rank(real_mean, placebo_col), 3),
        }
    )

print("Placebo permutation test — signed t+3 means")
print(pd.DataFrame(results).to_string(index=False))
print()
print("Headline: power signed rank = 0.98 — event days have systematically larger power")
print("moves than random non-event days over the same window. Brent and TTF signed means")
print("near placebo-median (rank 0.31–0.44) because event impacts go both directions.")
print()

# Histogram with real-event tick marker
fig = make_subplots(
    rows=1, cols=3, subplot_titles=("Brent t+3 mean", "TTF t+3 mean", "Power t+3 mean")
)
for i, (col, suffix, label) in enumerate(
    [
        ("brent_pct_3d", "brent", "Brent"),
        ("ttf_pct_3d", "ttf", "TTF"),
        ("power_pct_3d", "power", "Power"),
    ]
):
    placebo_col = placebo[f"{suffix}_pct_3d_mean"]
    real_mean = real[col].mean()
    fig.add_trace(
        go.Histogram(
            x=placebo_col, nbinsx=40, marker=dict(color="#888"), opacity=0.7, showlegend=False
        ),
        row=1,
        col=i + 1,
    )
    fig.add_vline(
        x=real_mean,
        line_color="#c0392b",
        line_width=3,
        annotation_text=f"real {real_mean:+.1f}%",
        annotation_position="top right",
        row=1,
        col=i + 1,
    )
fig.update_layout(
    title="Placebo distribution (1,000 iterations) vs real-event mean",
    template="plotly_white",
    height=320,
)
fig.show()

Placebo permutation test — signed t+3 means
metric (signed) real mean placebo median  rank
   brent_pct_3d    +1.49%         +1.75% 0.438
     ttf_pct_3d    +1.03%         +2.28% 0.307
   power_pct_3d   +13.90%         +2.85% 0.982

Headline: power signed rank = 0.98 — event days have systematically larger power
moves than random non-event days over the same window. Brent and TTF signed means
near placebo-median (rank 0.31–0.44) because event impacts go both directions.



### Phase-aware multi-horizon split (crude phases)

Each timeline event tagged with one of three phases — `war-risk repricing` (n=2), `supply destruction` (n=6), or `de-escalation` (n=7) — and per-event impacts rolled up per phase × horizon. Tests the PDF's prediction that phase 1 is visible at t+3, phase 2 needs t+10/t+20, phase 3 is a regime shift.

View: `v_event_impact_multi_phase`.

In [22]:
phase_impacts = con.execute("SELECT * FROM v_event_impact_multi_phase").df()
print(phase_impacts.to_string(index=False))
print()
print("PDF prediction read:")
print("• the methodology-hardening section (n=2) dominates: Brent +4.7% / TTF +15.8% / power +64.3% at t+3.")
print("• Supply destruction (n=6) means come out negative — supply impact was largely priced")
print("  after the war-start move; several events were followed by partial de-escalation rhetoric.")
print("• De-escalation (n=7) shows negative commodities at t+3 but positive power (+19.0%) —")
print(
    "  power's idiosyncratic vol around event clusters carries through even when commodities ease."
)
print()
print("Caveat: n=2 for phase 1 is too small for CIs; phase result is descriptive, not formal.")
print(
    "Phase tagging involves judgement calls (e.g. 04-02 in de-escalation despite hawkish rhetoric,"
)
print(
    "because the Brent close that day fell on US-pullback hopes — placement followed the market)."
)

             phase  n_events  brent_pct_1d_mean  brent_pct_3d_mean  brent_pct_5d_mean  brent_pct_10d_mean  ttf_pct_1d_mean  ttf_pct_3d_mean  ttf_pct_5d_mean  ttf_pct_10d_mean  power_pct_1d_mean  power_pct_3d_mean  power_pct_5d_mean  power_pct_10d_mean
war-risk repricing         2                0.0                4.7               14.6                15.6              0.0             15.8             17.0               9.4                0.0               64.3               51.3                 4.7
supply destruction         6               -1.1               -1.6               -1.8                 2.1              0.6             -1.8             -4.3              -2.5               -0.9               -2.3              -12.6               -14.2
     de-escalation         7               -3.8               -0.3                0.6                -1.8             -3.1             -1.4              1.1              -3.4                9.3               19.0               10.6             

---

## Clean Spark Spread overlay (D → E mechanism)

Operationalises the merit-order theory. When CCGT is the marginal generator in the GB stack, observed power price should track the theoretical CCGT margin:

```text
theoretical_css_gbp_mwh = ttf * heat_rate * eur_gbp + uka * emissions_factor
residual                = power_observed - theoretical_css
```

Constants in `src/iran_shock/config.py`: HEAT_RATE_CCGT=1.85, EMISSIONS_FACTOR_CCGT=0.36, EUR_GBP_RATE=0.85.

| # | Move | What it tests |
|---|---|---|
| 2.1 | UKA proxy ingest (KRBN-anchored) | Adds carbon cost to the merit-order calculation |
| 2.2 | Clean Spark Spread daily view | Computes theoretical CCGT margin for every day |
| 2.3 | High-gas vs low-gas residual | Mechanism check: residual small on high-gas days, wide on low-gas days |
| 2.4 | Dashboard panel | Observed vs theoretical + per-event residuals |

The expected result: the merit-order theory predicts observed power tracks theoretical CSS *when gas is marginal*, decouples when it isn't. The conditional-transmission finding from the conditional-transmission test becomes a direct mechanism check — not just "TTF↔power correlation tightens on high-gas days" but "*observed power tracks the theoretical CCGT margin on high-gas days*".

### UKA proxy (KRBN-anchored)

ICE-direct UKA needs a paid subscription. The free path is **KraneShares Global Carbon Strategy ETF (KRBN, NYSEARCA)** — basket of ICE carbon futures with the EU ETS (EUA) dominant weighting plus RGGI, CCA, and UK ETS. We anchor KRBN to a base UKA price of £72/tCO₂ on 2026-01-02 (a sensible early-2026 ICE UKA print) and let subsequent days move relative to KRBN. View: `v_uka_proxy_daily`. Fetcher: `fetch_uka_proxy` in `src/iran_shock/prices.py`.

In [23]:
uka = con.execute("""
    SELECT price_date, uka_close_gbp_per_tco2, krbn_close_usd
    FROM v_uka_proxy_daily ORDER BY price_date
""").df()
uka["price_date"] = pd.to_datetime(uka["price_date"])

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(
        x=uka["price_date"],
        y=uka["uka_close_gbp_per_tco2"],
        name="UKA proxy (£/tCO₂)",
        line=dict(color="#27ae60", width=2),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=uka["price_date"],
        y=uka["krbn_close_usd"],
        name="KRBN ETF (USD)",
        line=dict(color="#888", width=1.5, dash="dot"),
    ),
    secondary_y=True,
)
for _, e in timeline.iterrows():
    fig.add_vline(x=e["event_date"], line_dash="dot", line_color="grey", opacity=0.3)
fig.update_layout(
    title="UKA proxy = KRBN ETF rescaled to £72/tCO₂ on 2026-01-02",
    template="plotly_white",
    height=340,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(title_text="UKA proxy (£/tCO₂)", secondary_y=False)
fig.update_yaxes(title_text="KRBN NAV (USD)", secondary_y=True)
fig.show()

print(
    f"UKA proxy range: £{uka['uka_close_gbp_per_tco2'].min():.1f} – £{uka['uka_close_gbp_per_tco2'].max():.1f}/tCO₂, mean £{uka['uka_close_gbp_per_tco2'].mean():.1f}"
)
print("Anchor: £72/tCO₂ at 2026-01-02 (sensible early-2026 UKA spot level).")
print("KRBN tracks EU + UK + US + CA carbon futures (EUA-dominant); used as a UKA proxy.")

UKA proxy range: £55.0 – £73.4/tCO₂, mean £63.5
Anchor: £72/tCO₂ at 2026-01-02 (sensible early-2026 UKA spot level).
KRBN tracks EU + UK + US + CA carbon futures (EUA-dominant); used as a UKA proxy.


### CSS time-series + residual

Three things to read off the chart:

1. **Observed power (blue) and theoretical CSS (green dashed) should track on high-gas days.**
2. **Residual (red, right axis) should bounce around zero on high-gas days and diverge on low-gas days.**
3. **Event-day vertical lines** show which curated events landed in which regime.

View: `v_clean_spark_spread_daily`.

In [24]:
css = con.execute("SELECT * FROM v_clean_spark_spread_daily ORDER BY price_date").df()
css["price_date"] = pd.to_datetime(css["price_date"])

# Daily summary by regime
regime_summary = con.execute("""
    SELECT regime,
           COUNT(*) AS n,
           ROUND(AVG(power_observed_gbp_mwh), 1)   AS mean_power_obs,
           ROUND(AVG(theoretical_css_gbp_mwh), 1)  AS mean_theoretical,
           ROUND(AVG(residual_gbp_mwh), 1)         AS mean_residual,
           ROUND(AVG(ABS(residual_gbp_mwh)), 1)    AS mean_abs_residual,
           ROUND(STDDEV(residual_gbp_mwh), 1)      AS std_residual
    FROM v_clean_spark_spread_daily
    WHERE regime IN ('high_gas', 'low_gas')
    GROUP BY regime ORDER BY regime
""").df()
print("Daily CSS summary by regime (n=64 high-gas, n=38 low-gas):")
print(regime_summary.to_string(index=False))
print()

# Headline correlations
hi = css[css["regime"] == "high_gas"][["power_observed_gbp_mwh", "theoretical_css_gbp_mwh"]]
lo = css[css["regime"] == "low_gas"][["power_observed_gbp_mwh", "theoretical_css_gbp_mwh"]]
print("Observed power ↔ theoretical CSS, by regime:")
print(
    f"  high_gas (n={len(hi)}): Pearson r={hi.corr().iloc[0, 1]:+.3f}  Spearman ρ={hi.corr(method='spearman').iloc[0, 1]:+.3f}"
)
print(
    f"  low_gas  (n={len(lo)}): Pearson r={lo.corr().iloc[0, 1]:+.3f}  Spearman ρ={lo.corr(method='spearman').iloc[0, 1]:+.3f}"
)

# Time-series chart
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(
        x=css["price_date"],
        y=css["power_observed_gbp_mwh"],
        name="Observed power",
        line=dict(color="#2980b9", width=2),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=css["price_date"],
        y=css["theoretical_css_gbp_mwh"],
        name="Theoretical CSS",
        line=dict(color="#27ae60", width=2, dash="dash"),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=css["price_date"],
        y=css["residual_gbp_mwh"],
        name="Residual (obs − CSS)",
        line=dict(color="#c0392b", width=1.5),
    ),
    secondary_y=True,
)
for _, e in timeline.iterrows():
    fig.add_vline(x=e["event_date"], line_dash="dot", line_color="grey", opacity=0.3)
fig.update_layout(
    title="Observed GB power vs theoretical CCGT margin (£/MWh)",
    template="plotly_white",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(title_text="£/MWh", secondary_y=False)
fig.update_yaxes(title_text="Residual £/MWh", secondary_y=True)
fig.show()

# Residual scatter by regime
fig2 = px.scatter(
    css.dropna(subset=["theoretical_css_gbp_mwh", "residual_gbp_mwh"]),
    x="theoretical_css_gbp_mwh",
    y="power_observed_gbp_mwh",
    color="regime",
    color_discrete_map={"high_gas": "#c0392b", "low_gas": "#2980b9", "unknown": "#888"},
    labels={
        "theoretical_css_gbp_mwh": "Theoretical CSS (£/MWh)",
        "power_observed_gbp_mwh": "Observed power (£/MWh)",
    },
    title="Observed power vs theoretical CSS, by gas regime",
)
fig2.add_shape(type="line", x0=0, y0=0, x1=200, y1=200, line=dict(color="grey", dash="dot"))
fig2.update_layout(template="plotly_white", height=480)
fig2.show()

Daily CSS summary by regime (n=64 high-gas, n=38 low-gas):
  regime  n  mean_power_obs  mean_theoretical  mean_residual  mean_abs_residual  std_residual
high_gas 64           102.4              87.7           14.7               14.7          14.1
 low_gas 38            86.8              93.8           -6.9               13.2          17.2

Observed power ↔ theoretical CSS, by regime:
  high_gas (n=64): Pearson r=+0.608  Spearman ρ=+0.641
  low_gas  (n=38): Pearson r=+0.111  Spearman ρ=+0.094


### Per-event residuals + the mechanism read

For each timeline event, the residual = observed_power − theoretical_css on the event day, joined to the gas regime. View: `v_css_regime_event_impact`.

In [25]:
css_events = con.execute("""
    SELECT event_date, label, regime, gas_share_pct,
           ROUND(power_observed_gbp_mwh, 1)   AS power_obs,
           ROUND(theoretical_css_gbp_mwh, 1)  AS theoretical_css,
           ROUND(residual_gbp_mwh, 1)         AS residual
    FROM v_css_regime_event_impact
    ORDER BY event_date
""").df()
print("Per-event CSS residuals on event day:")
print(css_events.to_string(index=False))
print()
print("Read: on high-gas events, residuals are mostly small positive (£2–25/MWh) — observed")
print("power tracks the theoretical CCGT margin with a typical operating premium. On low-gas")
print("events, residuals are large negative (−£12 to −£48/MWh) — observed power sits well")
print("BELOW the theoretical CCGT margin because gas isn't actually setting the price")
print("(renewables / nuclear / imports are cheaper than the CCGT margin on those days).")
print()
print("D→E mechanism summary:")
print("• On high-gas days, gas IS marginal and observed power tracks theoretical CSS at r=+0.61.")
print("• On low-gas days, gas is NOT marginal — observed power decouples from CSS (r=+0.11).")
print("• The conditional-transmission finding from the conditional-transmission test isn't just empirical correlation;")
print("  it's the merit-order theory predicting itself in the data.")

Per-event CSS residuals on event day:
event_date                                            label   regime  gas_share_pct  power_obs  theoretical_css  residual
2026-02-28                       Operation Epic Fury begins  low_gas           23.3       72.2             91.3     -19.0
2026-03-01                   Iran retaliates; first US dead  low_gas           23.3       72.2             91.3     -19.0
2026-03-05                          Hormuz paralysis starts high_gas           35.8      107.3            100.9       6.3
2026-03-07    Israeli strikes hit Tehran/Alborz fuel depots high_gas           58.7      135.3            110.0      25.3
2026-03-12                   KC-135 loss and tanker attacks  low_gas           16.7       87.6            100.3     -12.7
2026-03-18            Iran strikes Qatar LNG infrastructure high_gas           38.5      110.0            106.0       4.0
2026-03-24                  US 15-point settlement proposal  low_gas           12.7       58.0            10

---

## Sentiment depth — A → B link (GDELT audit + GKG + FinBERT)

The headline finding so far rests on `AvgTone` — a single dictionary-based score. the sentiment-depth section audits the GDELT signal layer honestly and adds three richer sources:

| # | Move | What it adds |
|---|---|---|
| 3.1 | Goldstein × NumMentions + QuadClass + EventRootCode | Calibrated political-stability impact already in our DuckDB |
| 3.2 | FinBERT on SOURCEURL slugs | Finance-tuned sentiment per article URL |
| 3.3 | GKG ingest (every 6h, theme-filtered at ingest) | 6-part tone vector + V2THEMES + V2EnhancedLocations + GCAM |
| 3.4 | Energy themes + Hormuz / Persian Gulf location matching | Entity-resolved chokepoint signal |
| 3.5 | Cherry-picked GCAM dimensions | LIWC + GI + WordNetAffect emotional axes |
| 3.6 | Sentiment dispersion regime | Article-level variance per day as market-disagreement proxy |

See `docs/00_findings.md` for the consolidated numbers and the GDELT audit notes.

### GoldsteinScale × NumMentions vs AvgTone

GoldsteinScale is a CAMEO-event-type score in [-10, +10] calibrated by political scientists. NumMentions weights by coverage volume. The two combined produce a signal that is *less noisy than dictionary tone* for political events. View: `v_gdelt_events_features_daily`.

In [26]:
events_features = con.execute("""
    SELECT f.event_date, f.mean_tone, f.mentions_weighted_tone,
           f.mean_goldstein, f.mentions_weighted_goldstein,
           f.n_material_conflict, f.n_fight,
           b.brent_close, t.ttf_close, p.gb_power_avg
    FROM v_gdelt_events_features_daily f
    LEFT JOIN v_brent_daily b    ON b.price_date = f.event_date
    LEFT JOIN v_ttf_daily t      ON t.price_date = f.event_date
    LEFT JOIN v_gb_power_daily p ON p.price_date = f.event_date
    ORDER BY f.event_date
""").df()
events_features['brent_pct_3d'] = events_features['brent_close'].pct_change(3) * 100
events_features['ttf_pct_3d']   = events_features['ttf_close'].pct_change(3) * 100
events_features['power_pct_3d'] = events_features['gb_power_avg'].pct_change(3) * 100
clean = events_features.dropna(subset=['mean_goldstein','brent_pct_3d','ttf_pct_3d','power_pct_3d'])

print(f"n trading-day pairs: {len(clean)}")
print()
rows = []
for col in ['mean_tone', 'mentions_weighted_tone', 'mean_goldstein', 'mentions_weighted_goldstein']:
    rows.append({
        'signal':            col,
        '↔ brent_pct_3d':   round(clean[col].corr(clean['brent_pct_3d']), 3),
        '↔ ttf_pct_3d':     round(clean[col].corr(clean['ttf_pct_3d']), 3),
        '↔ power_pct_3d':   round(clean[col].corr(clean['power_pct_3d']), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Headline 3.1: mentions_weighted_goldstein ↔ TTF t+3 = -0.33, vs mean_tone -0.15.")
print("Goldstein × NumMentions is a 2× stronger TTF transmission signal than AvgTone,")
print("and it was already in our DuckDB — no new ingest required. The 'free zero-cost")
print("upgrade' from the the sentiment-depth section audit is real signal, not narrative.")

n trading-day pairs: 55



                     signal  ↔ brent_pct_3d  ↔ ttf_pct_3d  ↔ power_pct_3d
                  mean_tone          -0.102        -0.147          -0.226
     mentions_weighted_tone          -0.069        -0.139          -0.177
             mean_goldstein          -0.416        -0.329          -0.055
mentions_weighted_goldstein          -0.399        -0.334          -0.039

Headline 3.1: mentions_weighted_goldstein ↔ TTF t+3 = -0.33, vs mean_tone -0.15.
Goldstein × NumMentions is a 2× stronger TTF transmission signal than AvgTone,
and it was already in our DuckDB — no new ingest required. The 'free zero-cost
upgrade' from the Phase 3 audit is real signal, not narrative.


### FinBERT on event-window SOURCEURL slugs

We score the URL slug (last path segment) with FinBERT, not the full headline (which would need scraping). Coverage: top 200 URLs/day by NumMentions in a ±1-day window around each timeline event, deduplicated — 17,980 URLs total. CPU-time ~75 seconds.

View: `v_finbert_daily`. Module: `src/iran_shock/finbert.py`.

The key comparison is **AvgTone ↔ FinBERT correlation**: if they're tightly correlated, FinBERT is redundant. If they're only weakly correlated, FinBERT picks up a different signal.

In [27]:
fb = con.execute("""
    SELECT f.event_date, f.mean_finbert_signed, f.mentions_weighted_finbert_signed,
           f.finbert_dispersion, m.avg_tone
    FROM v_finbert_daily f
    JOIN v_middle_east_events m ON m.event_date = f.event_date
    ORDER BY f.event_date
""").df()
print(f"n days with both signals: {len(fb)}")
print(f"  AvgTone:         range [{fb['avg_tone'].min():.2f}, {fb['avg_tone'].max():.2f}]  mean {fb['avg_tone'].mean():.2f}")
print(f"  FinBERT signed:  range [{fb['mean_finbert_signed'].min():.3f}, {fb['mean_finbert_signed'].max():.3f}]  mean {fb['mean_finbert_signed'].mean():.3f}")
r = fb['avg_tone'].corr(fb['mean_finbert_signed'])
print(f"  AvgTone ↔ FinBERT signed Pearson r = {r:+.3f}")
print()
print("Read: r = +0.18 — weak correlation. FinBERT is measuring a different aspect")
print("of sentiment than the AvgTone dictionary score. The CV-named 'planned' move")
print("is shipped and produces orthogonal signal — not just a re-skin of AvgTone.")

# Side-by-side time-series
import plotly.graph_objects as go
fig = make_subplots(specs=[[{"secondary_y": True}]])
fb['event_date'] = pd.to_datetime(fb['event_date'])
fig.add_trace(go.Scatter(x=fb['event_date'], y=fb['avg_tone'],
                          name='AvgTone (Events)', line=dict(color='#888', width=2)),
              secondary_y=False)
fig.add_trace(go.Scatter(x=fb['event_date'], y=fb['mean_finbert_signed'],
                          name='FinBERT signed', line=dict(color='#c0392b', width=2)),
              secondary_y=True)
for _, e in timeline.iterrows():
    fig.add_vline(x=e['event_date'], line_dash='dot', line_color='grey', opacity=0.3)
fig.update_layout(title="AvgTone vs FinBERT signed — daily means",
                  template='plotly_white', height=380,
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.update_yaxes(title_text='AvgTone (-100..+100)', secondary_y=False)
fig.update_yaxes(title_text='FinBERT signed (-1..+1)', secondary_y=True)
fig.show()

n days with both signals: 91
  AvgTone:         range [-5.03, -3.89]  mean -4.59
  FinBERT signed:  range [-0.380, -0.129]  mean -0.236
  AvgTone ↔ FinBERT signed Pearson r = +0.181

Read: r = +0.18 — weak correlation. FinBERT is measuring a different aspect
of sentiment than the AvgTone dictionary score. The CV-named 'planned' move
is shipped and produces orthogonal signal — not just a re-skin of AvgTone.


### GKG 6-part tone vector

GKG's V2Tone has six components (positive, negative, polarity, activity-density, self-reference, plus the overall tone and word-count). We aggregated 115,915 GKG articles to 151 daily means. View: `v_gkg_tone_daily`.

In [28]:
gkg_tone = con.execute("""
    SELECT event_date, n_articles, mean_tone, mean_positive, mean_negative,
           mean_polarity, tone_dispersion
    FROM v_gkg_tone_daily ORDER BY event_date
""").df()
gkg_tone['event_date'] = pd.to_datetime(gkg_tone['event_date'])
print(f"n days: {len(gkg_tone)}, total articles: {gkg_tone['n_articles'].sum()}")
print()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("GKG 6-part tone components", "Tone dispersion (article-level std per day)"))
fig.add_trace(go.Scatter(x=gkg_tone['event_date'], y=gkg_tone['mean_positive'],
                          name='positive', line=dict(color='#27ae60')), row=1, col=1)
fig.add_trace(go.Scatter(x=gkg_tone['event_date'], y=gkg_tone['mean_negative'],
                          name='negative', line=dict(color='#c0392b')), row=1, col=1)
fig.add_trace(go.Scatter(x=gkg_tone['event_date'], y=gkg_tone['mean_polarity'],
                          name='polarity', line=dict(color='#8e44ad')), row=1, col=1)
fig.add_trace(go.Scatter(x=gkg_tone['event_date'], y=gkg_tone['tone_dispersion'],
                          name='tone std', line=dict(color='#2980b9')), row=2, col=1)
for _, e in timeline.iterrows():
    fig.add_vline(x=e['event_date'], line_dash='dot', line_color='grey', opacity=0.3)
fig.update_layout(template='plotly_white', height=520,
                  legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1))
fig.show()

n days: 151, total articles: 115915



### Energy themes + Hormuz / Persian Gulf entity mentions

GKG V2THEMES carry topic tags per article. We cherry-pick the energy + maritime + conflict themes. Note: **GDELT does NOT have a `MARITIME_CHOKEPOINT` theme** (we verified empirically — 0 rows). The free-text-search proxy is V2EnhancedLocations entity matches against `Hormuz`, `Persian Gulf`, `Kharg`, `Ras Laffan`, `Bandar`. War story #19 in `41_war_stories.md` has the audit.

Views: `v_gdelt_gkg_themes_daily`, `v_gdelt_gkg_hormuz_daily`.

In [29]:
themes = con.execute("""
    SELECT t.event_date, t.n_articles,
           t.n_econ_oil, t.n_energy_security, t.n_maritime_total, t.n_conflict_total,
           h.n_chokepoint_total, h.n_kharg, h.n_ras_laffan, h.n_persian_gulf
    FROM v_gdelt_gkg_themes_daily t
    LEFT JOIN v_gdelt_gkg_hormuz_daily h ON h.event_date = t.event_date
    ORDER BY t.event_date
""").df()
themes['event_date'] = pd.to_datetime(themes['event_date'])
print(f"n days: {len(themes)}, median oil-themed articles/day: {themes['n_econ_oil'].median():.0f}")
print()
print("Theme counts on key event dates:")
focus = themes[themes['event_date'].isin(pd.to_datetime(['2026-02-28','2026-03-05','2026-03-18','2026-04-13','2026-04-30']))]
print(focus[['event_date','n_econ_oil','n_maritime_total','n_chokepoint_total','n_kharg','n_ras_laffan']].to_string(index=False))
print()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=themes['event_date'], y=themes['n_econ_oil'],
                          name='ECON_OIL articles', line=dict(color='#e67e22', width=2)),
              secondary_y=False)
fig.add_trace(go.Scatter(x=themes['event_date'], y=themes['n_chokepoint_total'],
                          name='Chokepoint entity mentions', line=dict(color='#c0392b', width=2)),
              secondary_y=True)
for _, e in timeline.iterrows():
    fig.add_vline(x=e['event_date'], line_dash='dot', line_color='grey', opacity=0.3)
fig.update_layout(title="Energy themes + Hormuz/Persian Gulf entity mentions over time",
                  template='plotly_white', height=380,
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.update_yaxes(title_text='ECON_OIL article count', secondary_y=False)
fig.update_yaxes(title_text='Chokepoint mentions', secondary_y=True)
fig.show()

n days: 151, median oil-themed articles/day: 73

Theme counts on key event dates:
event_date  n_econ_oil  n_maritime_total  n_chokepoint_total  n_kharg  n_ras_laffan
2026-02-28        34.0             132.0                 7.0      4.0           0.0
2026-03-05       199.0             245.0                 4.0      0.0           4.0
2026-03-18       286.0             232.0                20.0     13.0           5.0
2026-04-13       186.0             357.0                11.0      5.0           6.0
2026-04-30       174.0             212.0                 3.0      2.0           0.0



### Cherry-picked GCAM emotional dimensions

GCAM has ~2,300 dimensions per article from LIWC, GI, WordNetAffect, and other dictionaries. We extract 8 picks relevant to energy + conflict + market response: LIWC anger / anxiety / certainty / negate; GI hostile / power; WordNetAffect fear / surprise. Reference: `data/reference/gcam_dimensions.csv`. View: `v_gdelt_gkg_gcam_daily`.

In [30]:
gcam = con.execute("""
    SELECT event_date, n_articles, mean_liwc_anger, mean_liwc_anxiety, mean_liwc_certainty,
           mean_gi_hostile, mean_wna_fear, mean_wna_surprise,
           dispersion_liwc_anxiety, dispersion_wna_fear
    FROM v_gdelt_gkg_gcam_daily ORDER BY event_date
""").df()
gcam['event_date'] = pd.to_datetime(gcam['event_date'])

# Three emotion axes over time
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('GCAM emotional dimensions',
                                    'Dispersion of LIWC anxiety and WNA fear — market-disagreement proxy (3.6)'))
fig.add_trace(go.Scatter(x=gcam['event_date'], y=gcam['mean_liwc_anger'],
                          name='LIWC anger', line=dict(color='#c0392b')), row=1, col=1)
fig.add_trace(go.Scatter(x=gcam['event_date'], y=gcam['mean_liwc_anxiety'],
                          name='LIWC anxiety', line=dict(color='#e67e22')), row=1, col=1)
fig.add_trace(go.Scatter(x=gcam['event_date'], y=gcam['mean_wna_fear'],
                          name='WNA fear', line=dict(color='#8e44ad')), row=1, col=1)
fig.add_trace(go.Scatter(x=gcam['event_date'], y=gcam['dispersion_liwc_anxiety'],
                          name='anxiety dispersion', line=dict(color='#e67e22', dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=gcam['event_date'], y=gcam['dispersion_wna_fear'],
                          name='fear dispersion', line=dict(color='#8e44ad', dash='dash')), row=2, col=1)
for _, e in timeline.iterrows():
    fig.add_vline(x=e['event_date'], line_dash='dot', line_color='grey', opacity=0.3)
fig.update_layout(template='plotly_white', height=600,
                  legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1))
fig.show()

print()
print("Read: emotion axes spike on supply-disruption events (5 Mar Hormuz paralysis,")
print("18 Mar Qatar LNG, 13 Apr US blockade). The dispersion overlay (3.6) tracks article-")
print("level variance — high dispersion days = disagreement / uncertainty regime.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Read: emotion axes spike on supply-disruption events (5 Mar Hormuz paralysis,
18 Mar Qatar LNG, 13 Apr US blockade). The dispersion overlay (3.6) tracks article-
level variance — high dispersion days = disagreement / uncertainty regime.


---

## Implied TTF / implied heat rate (mechanism inverse)

the CSS overlay asked: does observed power track theoretical CSS? the implied-TTF check asks the inverse: what TTF does observed power imply, and does it match what the spot market is pricing?

```text
implied_ttf_eur_mwh = (power_observed_gbp_mwh − uka × emissions_factor) / (heat_rate × eur_gbp)
basis_eur_mwh       = observed_ttf − implied_ttf
```

* Basis ≈ 0 → gas is plausibly marginal (spot TTF matches what power says marginal gas should cost).
* Basis > 0 → power is *cheaper* than implied gas — something else is at the margin (renewables / nuclear / imports).
* Basis < 0 → power is *more expensive* than implied gas — scarcity / operating premium on top.

View: `v_implied_ttf_daily`. Constants: same as the CSS overlay (`HEAT_RATE_CCGT=1.85`, `EMISSIONS_FACTOR_CCGT=0.36`, `EUR_GBP_RATE=0.85`).

In [31]:
implied = con.execute("""
    SELECT price_date, observed_ttf_eur_mwh, implied_ttf_eur_mwh,
           basis_eur_mwh, regime, gas_share_pct, basis_regime
    FROM v_implied_ttf_daily
    ORDER BY price_date
""").df()
implied['price_date'] = pd.to_datetime(implied['price_date'])

# Summary by regime
summary = con.execute("""
    SELECT regime,
           COUNT(*) AS n,
           ROUND(AVG(observed_ttf_eur_mwh), 1) AS mean_observed_ttf,
           ROUND(AVG(implied_ttf_eur_mwh), 1)  AS mean_implied_ttf,
           ROUND(AVG(basis_eur_mwh), 1)        AS mean_basis,
           ROUND(STDDEV(basis_eur_mwh), 1)     AS std_basis
    FROM v_implied_ttf_daily
    WHERE regime IN ('high_gas','low_gas')
    GROUP BY regime ORDER BY regime
""").df()
print("Implied TTF by gas regime:")
print(summary.to_string(index=False))
print()

# Correlations by regime
hi = implied[implied['regime'] == 'high_gas'][['observed_ttf_eur_mwh','implied_ttf_eur_mwh']]
lo = implied[implied['regime'] == 'low_gas'][['observed_ttf_eur_mwh','implied_ttf_eur_mwh']]
print("Observed TTF ↔ implied TTF (the inverse mechanism check):")
print(f"  high_gas (n={len(hi)}): Pearson r = {hi.corr().iloc[0,1]:+.3f}, Spearman ρ = {hi.corr(method='spearman').iloc[0,1]:+.3f}")
print(f"  low_gas  (n={len(lo)}): Pearson r = {lo.corr().iloc[0,1]:+.3f}, Spearman ρ = {lo.corr(method='spearman').iloc[0,1]:+.3f}")
print()

# Decoupling days
dc = implied[implied['basis_regime'] == 'decoupled']
print(f"Decoupling days (|basis| > €15/MWh): {len(dc)} / {len(implied)} ({100*len(dc)/len(implied):.1f}%)")
print(f"  in high_gas regime: {len(dc[dc['regime']=='high_gas'])}")
print(f"  in low_gas regime:  {len(dc[dc['regime']=='low_gas'])}")

Implied TTF by gas regime:
  regime  n  mean_observed_ttf  mean_implied_ttf  mean_basis  std_basis
high_gas 64               41.1              50.5        -9.3        9.0
 low_gas 38               45.3              40.9         4.4       11.0

Observed TTF ↔ implied TTF (the inverse mechanism check):
  high_gas (n=64): Pearson r = +0.627, Spearman ρ = +0.653
  low_gas  (n=38): Pearson r = +0.116, Spearman ρ = +0.085

Decoupling days (|basis| > €15/MWh): 12 / 102 (11.8%)
  in high_gas regime: 7
  in low_gas regime:  5


In [32]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Observed TTF (€/MWh) vs implied TTF from observed power",
                                    "Basis = observed − implied (€/MWh); |basis|>€15 highlighted"))
fig.add_trace(go.Scatter(x=implied['price_date'], y=implied['observed_ttf_eur_mwh'],
                          name='Observed TTF', line=dict(color='#27ae60', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=implied['price_date'], y=implied['implied_ttf_eur_mwh'],
                          name='Implied TTF', line=dict(color='#c0392b', width=2, dash='dash')), row=1, col=1)
fig.add_trace(go.Bar(x=implied['price_date'], y=implied['basis_eur_mwh'],
                      name='Basis',
                      marker_color=['#888' if r=='plausibly_marginal' else '#e67e22'
                                    for r in implied['basis_regime']]),
              row=2, col=1)
fig.add_hline(y=15,  line_dash='dot', line_color='red', opacity=0.5, row=2, col=1)
fig.add_hline(y=-15, line_dash='dot', line_color='red', opacity=0.5, row=2, col=1)
for _, e in timeline.iterrows():
    fig.add_vline(x=e['event_date'], line_dash='dot', line_color='grey', opacity=0.3)
fig.update_layout(template='plotly_white', height=620,
                  legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1))
fig.show()

# Scatter
fig2 = px.scatter(implied.dropna(subset=['observed_ttf_eur_mwh','implied_ttf_eur_mwh']),
                  x='implied_ttf_eur_mwh', y='observed_ttf_eur_mwh',
                  color='regime',
                  color_discrete_map={'high_gas':'#c0392b','low_gas':'#2980b9','unknown':'#888'},
                  size='gas_share_pct',
                  labels={'implied_ttf_eur_mwh':'Implied TTF (from observed power)',
                          'observed_ttf_eur_mwh':'Observed TTF (spot)'},
                  title='Observed vs implied TTF — only correlated when gas is marginal')
fig2.add_shape(type='line', x0=0, y0=0, x1=120, y1=120, line=dict(color='grey', dash='dot'))
fig2.update_layout(template='plotly_white', height=480)
fig2.show()

print()
print("the implied-TTF check mechanism summary — the cleanest single sentence:")
print("• On high-gas days, observed TTF tracks implied TTF at Pearson r=+0.63.")
print("• On low-gas days, the correlation collapses to r=+0.12.")
print("• This is the CSS overlay's merit-order finding viewed from the inverse direction:")
print("  the CSS overlay said observed power tracks theoretical CSS when gas is marginal;")
print("  the implied-TTF check says observed gas tracks power-implied gas under the same condition.")
print("• ~12% of days had |basis|>€15 — the merit-order model didn't hold on those days.")


Phase 4 mechanism summary — the cleanest single sentence:
• On high-gas days, observed TTF tracks implied TTF at Pearson r=+0.63.
• On low-gas days, the correlation collapses to r=+0.12.
• This is Phase 2's merit-order finding viewed from the inverse direction:
  Phase 2 said observed power tracks theoretical CSS when gas is marginal;
  Phase 4 says observed gas tracks power-implied gas under the same condition.
• ~12% of days had |basis|>€15 — the merit-order model didn't hold on those days.


## Export to static HTML (for GitHub Pages)

Optional: write the rendered notebook to a single HTML file under `docs-site/`. A GHA workflow can re-execute this notebook on a `schedule:` cron and publish the output to GitHub Pages. See `docs/22_migrate_to_aws.md` for the workflow YAML.

In [33]:
# To export: from the project root, run
#   uv run jupyter nbconvert --execute notebooks/06_dashboard.ipynb \
#       --to html --output-dir docs-site/
# Then commit docs-site/06_dashboard.html for GitHub Pages.